In [1]:
import sqlite3
import pandas as pd

# Verbindung zur Datenbank aufbauen
db_path = "../data/viessmann_events - Kopie.db"
conn = sqlite3.connect(db_path)

# Alle Tabellennamen abfragen
query = "SELECT name FROM sqlite_master WHERE type='table';"
tabellen = pd.read_sql_query(query, conn)

# Verbindung schließen
conn.close()

# Tabellen anzeigen
print("Folgende Tabellen wurden in der Datenbank gefunden:")
print(tabellen)

Folgende Tabellen wurden in der Datenbank gefunden:
                       name
0                    events
1           sqlite_sequence
2     temperature_snapshots
3  temperature_log_settings
4         schema_migrations


In [2]:
# Daten einsehen
import sqlite3
import pandas as pd

# Pfad zu deiner Datenbank
db_path = "../data/viessmann_events - Kopie.db"
conn = sqlite3.connect(db_path)

# Die ersten 10 Zeilen aus den Snapshots abfragen
query_snapshots = "SELECT * FROM temperature_snapshots LIMIT 10;"
df_snapshots = pd.read_sql_query(query_snapshots, conn)

conn.close()

# Daten im Notebook schön anzeigen lassen
df_snapshots

,id,timestamp,installation_id,gateway_id,device_id,account_id,account_name,outside_temp,return_temp,supply_temp,...,heating_circuit_2_supply_temp,heating_circuit_3_supply_temp,heating_circuit_0_delta_t,heating_circuit_1_delta_t,heating_circuit_2_delta_t,heating_circuit_3_delta_t,compressor_starts,four_way_valve_current,four_way_valve_target,pressure_supply
0,2,2026-09-04T14:35:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,39.5,None,...,None,None,6.7,None,None,None,2756.0,100.0,100.0,1.9
1,4,2026-09-04T14:40:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,39.5,None,...,None,None,6.7,None,None,None,2756.0,100.0,100.0,1.9
2,6,2026-09-04T14:45:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,40.5,None,...,None,None,6.7,None,None,None,2756.0,100.0,100.0,1.9
3,8,2026-09-04T14:50:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,40.5,None,...,None,None,6.7,None,None,None,2756.0,100.0,100.0,1.9
4,10,2026-09-04T14:55:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,41.5,None,...,None,None,6.7,None,None,None,2756.0,100.0,100.0,1.9
5,11,2026-09-04T14:59:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,41.5,None,...,None,None,6.7,None,None,None,2756.0,100.0,100.0,1.9
6,12,2026-09-04T15:00:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,41.5,None,...,None,None,6.7,None,None,None,2756.0,100.0,100.0,1.9
7,14,2026-09-04T15:05:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,43.0,None,...,None,None,6.6,None,None,None,2756.0,100.0,100.0,1.9
8,15,2026-09-04T15:09:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,43.0,None,...,None,None,6.6,None,None,None,2756.0,100.0,100.0,1.9
9,16,2026-09-04T15:10:00Z,2940869,7952592009769246,0,manfred.sommer@kabelmail.de,manfred.sommer@kabelmail.de,29.5,43.0,None,...,None,None,6.6,None,None,None,2756.0,100.0,100.0,1.9


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../data/viessmann_events - Kopie.db")

# Wir laden die wichtigsten Spalten für die Analyse
query = """
SELECT timestamp, outside_temp, supply_temp, compressor_active 
FROM temperature_snapshots;
"""
df = pd.read_sql_query(query, conn)
conn.close()

# 1. Zeitstempel in echte Datums-Objekte umwandeln
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Zeitdifferenz zwischen aufeinanderfolgenden Datenpunkten berechnen (in Minuten)
df = df.sort_values('timestamp')
df['time_diff_min'] = df['timestamp'].diff().dt.total_seconds() / 60.0

# 3. Große Zeitlücken finden (z.B. alles was größer als 7 Minuten ist)
lücken = df[df['time_diff_min'] > 7.0]

print(f"Gesamtanzahl der Datenpunkte: {len(df)}")
print(f"Anzahl gefundener Zeitlücken (> 7 Min): {len(lücken)}")
print("\nDie 5 größten Zeitlücken (potenzielle WLAN-Ausfälle):")
print(lücken[['timestamp', 'time_diff_min']].sort_values('time_diff_min', ascending=False).head(5))

# 4. Prüfen, ob es Zeilen mit komplett fehlenden Werten (Null/NaN) gibt
print("\nFehlende Werte (NaN) in den Sensoren:")
print(df[['outside_temp', 'supply_temp']].isna().sum())

DatabaseError: database disk image is malformed

Beobachtung: Beim Auslesen der gesamten Tabelle tritt ein DatabaseError: database disk image is malformed auf. Ursache ist vermutlich ein korrupter Index durch das Kopieren im laufenden Betrieb. Nächster Schritt: SQLite Recovery-Dump durchführen.